# Nilufer Bina Metrekare Birim Degerleri - Datalake Upload

Bu notebook Databricks'te calisir. `Nilufer Bina Verisi Indir (Local).ipynb`
ile lokalde indirilip Databricks'e (DBFS/Volume) manuel yuklenen CSV'yi okur
ve data lake'e (bronze katman) yazar.

Kullanim: Notebook'u calistirmadan once ust kisimdaki `source_path` widget'ina
yukledigin CSV'nin Databricks path'ini yaz (orn. `dbfs:/FileStore/nilufer/nilufer_bina_birim_degerleri.csv`
veya bir Unity Catalog Volume path'i).

In [ ]:
%run "./Utils"

In [ ]:
dbutils.widgets.text("source_path", "", "Databricks source path (csv)")

SOURCE_PATH = dbutils.widgets.get("source_path")

if not SOURCE_PATH:
    raise ValueError("source_path widget'i bos. Yukledigin CSV'nin Databricks path'ini gir.")

print("Kaynak path:", SOURCE_PATH)

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("yil", IntegerType(), True),
    StructField("insaat_tur_adi", StringType(), True),
    StructField("insaat_sinif_adi", StringType(), True),
    StructField("insaat_sekli_adi", StringType(), True),
    StructField("bina_birim_degeri", DoubleType(), True),
])

df_spark = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(SOURCE_PATH)
)

print("Partition sayisi:", df_spark.rdd.getNumPartitions())
display(df_spark.limit(10))

In [ ]:
write_to_datalake(
    df_spark,
    "abfss://axetproject@ozandatalake001.dfs.core.windows.net/axet_bronze/nilufer_bina_birim_degerleri/"
)